## Data Ingestion (Streaming)

Here, we store the streaming data coming from **Kafka** as aggregated data in our Landing Zone.

**Importing Useful Libraries**

In [10]:
from kafka import KafkaConsumer
from dotenv import load_dotenv
import json
import boto3
import io
import os
import time

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [8]:
# -------------------------
# Kafka Consumers
# -------------------------
consumer_weather = KafkaConsumer(
    'weather-barcelona',
    bootstrap_servers='kafka:9092',
    value_deserializer=lambda v: json.loads(v),
    auto_offset_reset='earliest',
    enable_auto_commit=True
)
consumer_air = KafkaConsumer(
    'airquality-barcelona',
    bootstrap_servers='kafka:9092',
    value_deserializer=lambda v: json.loads(v),
    auto_offset_reset='earliest',
    enable_auto_commit=True
)

In [5]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [7]:
# Create two sub-buckets inside persistent-landing
s3.put_object(Bucket="landing-zone", Key="persistent-landing/weather-barcelona/")
s3.put_object(Bucket="landing-zone", Key="persistent-landing/airquality-barcelona/")

{'ResponseMetadata': {'RequestId': '189D9B5408BD4391',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': 'AAAAAA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '189D9B5408BD4391',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '9030',
   'x-ratelimit-remaining': '9030',
   'x-xss-protection': '1; mode=block',
   'date': 'Tue, 17 Mar 2026 10:50:13 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
 'ChecksumCRC32': 'AAAAAA==',
 'ChecksumType': 'FULL_OBJECT'}

In [11]:
# -------------------------
# Aggregation loop
# -------------------------

bucket_name = 'landing-zone'

while True:
    timestamp = int(time.time())

    # ----- Weather -----
    weather_rows = []
    for _ in range(10):
        msg = next(consumer_weather)
        weather_rows.append(msg.value)

    # Convert to JSON string in memory
    weather_buffer = io.StringIO()
    json.dump(weather_rows, weather_buffer, ensure_ascii=False)

    weather_key = f'persistent-landing/weather-barcelona/weather_{timestamp}.json'
    s3.put_object(
        Bucket=bucket_name,
        Key=weather_key,
        Body=weather_buffer.getvalue()
    )
    print(f"Uploaded weather data: {weather_key}")

    # ----- Air Quality -----
    air_rows = []
    for _ in range(10):
        msg = next(consumer_air)
        air_rows.append(msg.value)

    air_buffer = io.StringIO()
    json.dump(air_rows, air_buffer, ensure_ascii=False)

    air_key = f'persistent-landing/airquality-barcelona/airquality_{timestamp}.json'
    s3.put_object(
        Bucket=bucket_name,
        Key=air_key,
        Body=air_buffer.getvalue()
    )
    print(f"Uploaded air quality data: {air_key}")

    # Wait 1 minute before next batch
    time.sleep(60)

Uploaded weather data: persistent-landing/weather-barcelona/weather_1773745252.json
Uploaded air quality data: persistent-landing/airquality-barcelona/airquality_1773745252.json


KeyboardInterrupt: 